In [15]:
import fiftyone.zoo as foz
import os, json, shutil
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:


# ========== SETUP ==========
out_dir = "/home/parasite/backgrounds_clean"
os.makedirs(out_dir, exist_ok=True)

# ========== 1️DOWNLOAD COCO ==========
info, dataset_dir = foz.download_zoo_dataset(
    "coco-2017",
    split="validation",
    max_samples=10000,
)

ann_path = os.path.join(dataset_dir, "raw", "instances_val2017.json")
with open(ann_path, "r") as f:
    anns = json.load(f)

# Map category_id → name
cat_map = {c["id"]: c["name"] for c in anns["categories"]}

# Collect image_ids that contain a person
person_ids = {ann["image_id"] for ann in anns["annotations"]
              if cat_map.get(ann["category_id"]) == "person"}

# Copy only images without people
img_dir = os.path.join(dataset_dir, "validation", "data")
kept_files = []
for img_info in tqdm(anns["images"], desc="Filtering backgrounds (COCO annotations)"):
    if img_info["id"] in person_ids:
        continue
    src = os.path.join(img_dir, img_info["file_name"])
    dst = os.path.join(out_dir, img_info["file_name"])
    if os.path.exists(src):
        shutil.copy(src, dst)
        kept_files.append(dst)

print(f"Copied {len(kept_files)} images (annotation filter)")

# ========== 2️SANITY CHECK USING YOLO ==========
print("Running YOLO sanity check to remove any remaining people...")

model = YOLO("yolov8n.pt")  # lightweight pretrained detector
removed = 0

for img_path in tqdm(kept_files, desc="Verifying with YOLO"):
    results = model.predict(source=img_path, classes=[0], conf=0.25, verbose=False)
    # YOLO class 0 = person
    if any(len(r.boxes) > 0 for r in results):
        os.remove(img_path)
        removed += 1

print(f"Removed {removed} images with detected people.")
print(f"Final background count: {len(kept_files) - removed}")
print(f"Clean backgrounds stored in: {out_dir}")


Found annotations at '/home/parasite/fiftyone/coco-2017/raw/instances_val2017.json'


Only found 5000 (<10000) samples matching your requirements
2000 images found; downloading the remaining 3000
 100% |████████████████| 3000/3000 [7.7m elapsed, 0s remaining, 5.8 images/s]      
Writing annotations to '/home/parasite/fiftyone/coco-2017/validation/labels.json'
Dataset info written to '/home/parasite/fiftyone/coco-2017/info.json'


Filtering backgrounds (COCO annotations): 100%|██████████| 5000/5000 [00:01<00:00, 3844.29it/s]


Copied 2307 images (annotation filter)
Running YOLO sanity check to remove any remaining people...


Verifying with YOLO: 100%|██████████| 2307/2307 [00:18<00:00, 127.70it/s]

Removed 77 images with detected people.
Final background count: 2230
Clean backgrounds stored in: /home/parasite/backgrounds_clean


In [14]:
counts = [len(os.listdir(os.path.join('./backgrounds_clean')))]
print(counts)

[2230]


In [16]:

src_root = "./backgrounds_clean"         # your Places365 root directory
dst_root = "./backgrounds_clean" # flattened output directory

os.makedirs(dst_root, exist_ok=True)

count = 0
for root, _, files in tqdm(os.walk(src_root), desc="Flattening dataset"):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            src = os.path.join(root, file)
            dst = os.path.join(dst_root, file)

            # prevent overwriting if file already exists
            if os.path.exists(dst):
                name, ext = os.path.splitext(file)
                i = 1
                while True:
                    new_name = f"{name}_{i}{ext}"
                    new_dst = os.path.join(dst_root, new_name)
                    if not os.path.exists(new_dst):
                        dst = new_dst
                        break
                    i += 1

            shutil.copy2(src, dst)
            count += 1

print(f"\nFlattened {count} images into {dst_root}")


Flattening dataset: 366it [00:19, 18.63it/s]


Flattened 46643 images into ./backgrounds_clean
